In [1]:
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem import LancasterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

from transformers import BertTokenizer, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
data = pd.read_csv('/content/drive/MyDrive/NLPMiniProject_data.csv')
data

,ID,comment,date,down,parent_comment,score,top,topic,user,label
0,uid_590555,"Well, let's be honest here, they don't actuall...",2015-04,0,They should shut the fuck up and let the commu...,2,2,starcitizen,Combat_Wombatz,0
1,uid_671762,"Well, I didn't need evidence to believe in com...",2016-12,-1,You need evidence to kill people? I thought we...,6,-1,EnoughCommieSpam,starkadd,1
2,uid_519689,"Who does an ""official promo"" in 360p?",2013-11,0,2014 BMW S1000R: Official Promo,3,3,motorcycles,phybere,0
3,uid_788362,Grotto koth was the best,2015-09,0,Not really that memorable lol if you want memo...,2,2,hcfactions,m0xyMC,1
4,uid_299252,Neal's back baby,2015-11,0,James Neal hit on Zach Parise,-5,-5,hockey,Somuch101,1
...,...,...,...,...,...,...,...,...,...,...
14995,uid_845344,Well with a name like El Cubano I'm surprised ...,2015-01,0,There's two things you don't do in Florida. - ...,18,18,hockey,shutupisaac,0
14996,uid_757880,... This is a good point.,2014-04,0,Sounds like a pretty good overall summary of o...,6,6,hockey,em483,0
14997,uid_724706,Yep.,2015-09,0,"I know the type you speak of. The ""die cis scu...",2,2,AskReddit,YoImAli,0
14998,uid_1006984,That's what the government WANTS you to believe!,2016-01,0,That there's A hidden cure for cancer but phar...,1,1,AskReddit,OhHiGCHQ,1


In [3]:
combined_cleaned_text = data['comment'] + ' ' + data['parent_comment']
X1 = combined_cleaned_text
y1 = data['label'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

In [4]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(list(X_train.values), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test.values), truncation=True, padding=True, max_length=128)

train_dataset = TensorDataset(
    torch.tensor(train_encodings['input_ids']),
    torch.tensor(train_encodings['attention_mask']),
    torch.tensor(y_train)
)
test_dataset = TensorDataset(
    torch.tensor(test_encodings['input_ids']),
    torch.tensor(test_encodings['attention_mask']),
    torch.tensor(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

#  BERT Model Training with Tuned Hyperparameters
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("\n--- Fine-tuning BERT with Tuned Hyperparameters ---")
for epoch in range(5):
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} completed, Loss: {loss.item():.4f}")

#  BERT Model Evaluation
model.eval()
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        total_predictions += labels.size(0)
        correct_predictions += (predictions == labels).sum().item()

accuracy = correct_predictions / total_predictions
print(f"\nBERT Test Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Fine-tuning BERT with Tuned Hyperparameters ---
Epoch 1 completed, Loss: 0.7361
Epoch 2 completed, Loss: 0.4274
Epoch 3 completed, Loss: 0.3220
Epoch 4 completed, Loss: 0.0637
Epoch 5 completed, Loss: 0.0164

BERT Test Accuracy: 0.6810


In [5]:
#  RoBERTa Tokenization
tokenizer_roberta = RobertaTokenizer.from_pretrained('roberta-base')

train_encodings_roberta = tokenizer_roberta(list(X_train.values), truncation=True, padding=True, max_length=128)
test_encodings_roberta = tokenizer_roberta(list(X_test.values), truncation=True, padding=True, max_length=128)

train_dataset_roberta = TensorDataset(
    torch.tensor(train_encodings_roberta['input_ids']),
    torch.tensor(train_encodings_roberta['attention_mask']),
    torch.tensor(y_train)
)
test_dataset_roberta = TensorDataset(
    torch.tensor(test_encodings_roberta['input_ids']),
    torch.tensor(test_encodings_roberta['attention_mask']),
    torch.tensor(y_test)
)

train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=16, shuffle=True)
test_loader_roberta = DataLoader(test_dataset_roberta, batch_size=16, shuffle=False)

# RoBERTa Model Training and Evaluation
model_roberta = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
model_roberta.to(device)
model_roberta.train()

optimizer_roberta = torch.optim.AdamW(model_roberta.parameters(), lr=5e-5)

print("\n--- Fine-tuning RoBERTa on the Sarcasm Detection Dataset ---")
for epoch in range(5):
    for batch in train_loader_roberta:
        optimizer_roberta.zero_grad()
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model_roberta(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer_roberta.step()
    print(f"Epoch {epoch+1} completed, Loss: {loss.item():.4f}")

model_roberta.eval()
correct_predictions_roberta = 0
total_predictions_roberta = 0

with torch.no_grad():
    for batch in test_loader_roberta:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model_roberta(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        total_predictions_roberta += labels.size(0)
        correct_predictions_roberta += (predictions == labels).sum().item()

accuracy_roberta = correct_predictions_roberta / total_predictions_roberta
print(f"\nRoBERTa Test Accuracy: {accuracy_roberta:.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Fine-tuning RoBERTa on the Sarcasm Detection Dataset ---
Epoch 1 completed, Loss: 0.6735
Epoch 2 completed, Loss: 0.6968
Epoch 3 completed, Loss: 0.6888
Epoch 4 completed, Loss: 0.7204
Epoch 5 completed, Loss: 0.6974

RoBERTa Test Accuracy: 0.5017
